# Customer Segmentation using Fuzzy C-Means

This notebook implements customer segmentation using the Fuzzy C-Means clustering algorithm with correlation-based feature weighting.


In [1]:
import numpy as np
import pandas as pd
import skfuzzy as fuzz
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
import matplotlib.pyplot as plt
import seaborn as sns
import pickle


## Step 2: Load Customer Data


In [28]:
# Load customer features data
customer_data = pd.read_csv('customer_features_enhanced.csv')
print(f"Data shape: {customer_data.shape}")
print(f"Columns: {list(customer_data.columns)}")
print("\nFirst few rows:")
customer_data.head()

# Check for duplicate columns
print(f"\nChecking for duplicate columns...")
print(f"Has duplicate columns: {customer_data.columns.duplicated().any()}")
duplicate_columns = customer_data.columns[customer_data.columns.duplicated()]
print(f"Duplicate column names: {list(duplicate_columns)}")

# Show column counts
from collections import Counter
column_counts = Counter(customer_data.columns)
duplicated_with_counts = {col: count for col, count in column_counts.items() if count > 1}
print(f"Columns appearing multiple times: {duplicated_with_counts}")


Data shape: (1077, 69)
Columns: ['CustomerID', 'ExtPrice_sum', 'ExtPrice_mean', 'ExtPrice_std', 'ExtPrice_count', 'ExtPrice_median', 'ExtPrice_min', 'ExtPrice_max', 'ExtCost_sum', 'ExtCost_mean', 'ExtCost_std', 'ExtCost_median', 'SalesQty_sum', 'SalesQty_mean', 'SalesQty_std', 'SalesQty_median', 'SalesQty_min', 'SalesQty_max', 'UnitPrice_mean_x', 'UnitPrice_std_x', 'UnitPrice_min_x', 'UnitPrice_max_x', 'UnitPrice_median', 'UnitCost_mean', 'UnitCost_std', 'UnitCost_median', 'OrderNumber_nunique', 'ProductID_nunique', 'SalesDate_min', 'SalesDate_max', 'SalesDate_count', 'TotalRevenue', 'TotalCost', 'TotalProfit', 'ProfitMargin', 'AvgOrderValue', 'AvgOrderQuantity', 'PriceVolatility', 'QuantityVolatility', 'CustomerLifetimeDays', 'AvgDaysBetweenOrders', 'OrderFrequency', 'ProductDiversity', 'AvgProductsPerOrder', 'Recency', 'Frequency', 'Monetary', 'RecencyScore', 'FrequencyScore', 'MonetaryScore', 'RFMScore', 'CustomerLifetimeValue', 'CustomerLoyaltyScore', 'QuoteID_nunique', 'QuoteVersi

## Step 3: Pure Correlation Method for Feature Weights

This method determines feature weights based purely on statistical correlations with TotalProfit, without any business assumptions.


In [29]:
# Define features for segmentation (excluding TotalProfit as it's our target)
# Updated for enhanced dataset - using CustomerLifetimeDays instead of CustomerLifetime
features = ['ProfitMargin', 'AvgOrderValue', 'CustomerLifetimeDays', 
           'QuoteConversionRate', 'OrderNumber_nunique', 'ProductID_nunique', 'PriceVolatility']

print("Selected features for clustering:")
for i, feature in enumerate(features, 1):
    print(f"{i}. {feature}")




Selected features for clustering:
1. ProfitMargin
2. AvgOrderValue
3. CustomerLifetimeDays
4. QuoteConversionRate
5. OrderNumber_nunique
6. ProductID_nunique
7. PriceVolatility


In [30]:
# Step 1: Calculate raw correlations with TotalProfit
print("Step 1: Calculating correlations with TotalProfit")
correlations = customer_data[features].corrwith(customer_data['TotalProfit'])

print("\nRaw Correlations:")
for feature, corr in zip(features, correlations):
    print(f"{feature}: {corr:.4f}")

# Step 2: Use absolute values to handle negative correlations
print("\nStep 2: Using absolute correlations")
absolute_correlations = abs(correlations)

print("\nAbsolute Correlations:")
for feature, corr in zip(features, absolute_correlations):
    print(f"{feature}: {corr:.4f}")

# Step 3: Normalize weights to sum to 1
print("\nStep 3: Normalizing weights")
weights = absolute_correlations / absolute_correlations.sum()

print("\nFinal Feature Weights (Pure Correlation Method):")
for feature, weight in zip(features, weights):
    print(f"{feature}: {weight:.4f}")

print(f"\nWeights sum to: {weights.sum():.6f}")
print(f"Weight range: {weights.min():.4f} - {weights.max():.4f}")


Step 1: Calculating correlations with TotalProfit

Raw Correlations:
ProfitMargin: 0.1598
AvgOrderValue: 0.0838
CustomerLifetimeDays: 0.1390
QuoteConversionRate: -0.0212
OrderNumber_nunique: 0.8139
ProductID_nunique: 0.5101
PriceVolatility: 0.1721

Step 2: Using absolute correlations

Absolute Correlations:
ProfitMargin: 0.1598
AvgOrderValue: 0.0838
CustomerLifetimeDays: 0.1390
QuoteConversionRate: 0.0212
OrderNumber_nunique: 0.8139
ProductID_nunique: 0.5101
PriceVolatility: 0.1721

Step 3: Normalizing weights

Final Feature Weights (Pure Correlation Method):
ProfitMargin: 0.0841
AvgOrderValue: 0.0441
CustomerLifetimeDays: 0.0732
QuoteConversionRate: 0.0112
OrderNumber_nunique: 0.4284
ProductID_nunique: 0.2685
PriceVolatility: 0.0906

Weights sum to: 1.000000
Weight range: 0.0112 - 0.4284


## Step 4: Data Preprocessing


In [31]:
# Remove outliers (same logic as original model)
print("Removing outliers...")
print(f"Original data shape: {customer_data.shape}")

# Filter customers with TotalProfit between -300k and 3M
outlier_mask = (customer_data['TotalProfit'] < -300000) | (customer_data['TotalProfit'] > 3000000)
outliers = customer_data[outlier_mask]

print(f"Number of outliers found: {len(outliers)}")
if len(outliers) > 0:
    print("Outliers:")
    print(outliers[['CustomerID', 'TotalProfit']].head())

# Remove outliers
customer_data_clean = customer_data[~outlier_mask].copy()
print(f"Data shape after removing outliers: {customer_data_clean.shape}")
print(f"Removed {len(customer_data) - len(customer_data_clean)} customers")


Removing outliers...
Original data shape: (1077, 69)
Number of outliers found: 7
Outliers:
    CustomerID  TotalProfit
0        CUST1  7671724.424
1       CUST10  9276478.230
234     CUST24  3825169.810
411      CUST4  4197134.010
500     CUST48  3061938.332
Data shape after removing outliers: (1070, 69)
Removed 7 customers


In [32]:
# Prepare features for clustering
print("Preparing features for clustering...")

# Select only the features we need
X = customer_data_clean[features].copy()



print(f"Final feature matrix shape: {X.shape}")
print("Feature statistics:")
X.describe()


Preparing features for clustering...
Final feature matrix shape: (1070, 7)
Feature statistics:


,ProfitMargin,AvgOrderValue,CustomerLifetimeDays,QuoteConversionRate,OrderNumber_nunique,ProductID_nunique,PriceVolatility
count,1070.000000,1070.000000,1070.000000,1070.000000,1070.000000,1070.000000,1070.000000
mean,0.354810,1027.714098,796.353271,0.910966,165.384112,64.985981,1.582574
std,0.775169,2678.389737,492.985651,0.195303,832.733049,214.829688,2.052168
min,-23.095413,0.580000,0.000000,0.333333,1.000000,1.000000,0.000000
25%,0.287771,180.475962,337.000000,0.841998,4.000000,3.000000,0.478556
50%,0.391830,382.573516,949.000000,0.888210,13.000000,10.000000,1.083583
75%,0.484910,891.046086,1262.750000,0.954545,53.750000,48.000000,2.052955
max,1.000000,38179.468240,1424.000000,2.000000,17036.000000,2959.000000,31.383337


In [33]:
# Standardize features
print("Standardizing features...")
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(X_scaled[:5])


weights_array = np.array(weights)
X_weighted = X_scaled * weights_array





Standardizing features...
[[-4.65682749e-02  2.51344367e-03  1.06066196e+00 -1.00841325e-01
  -1.08589875e-01  3.53999323e-01  2.94968592e+00]
 [ 4.78911877e-01 -3.79879241e-01  1.01601505e+00  3.01741879e+00
  -1.96294015e-01 -2.93327404e-01 -4.03165810e-01]
 [ 2.01729173e-01 -3.76825606e-01 -1.61612343e+00 -2.10523518e+00
  -1.97495441e-01 -2.97984430e-01 -7.71532570e-01]
 [ 3.73010930e-01 -2.48666349e-01 -1.54951721e-01  4.56091806e-01
  -1.95092588e-01 -2.97984430e-01 -5.59440369e-01]
 [ 1.91808694e-01 -3.05353755e-01 -1.18994835e+00  3.01741879e+00
  -1.96294015e-01 -2.97984430e-01 -7.71532570e-01]]


## Step 5: Fuzzy C-Means Clustering


In [34]:
# Set clustering parameters
num_clusters = 6
fuzziness_parameter = 2

print(f"Running Fuzzy C-Means with {num_clusters} clusters...")
print(f"Fuzziness parameter: {fuzziness_parameter}")

# Perform fuzzy clustering
center, u, u0, distance, cost, partition_coeff, fuzz_partition_coeff = fuzz.cluster.cmeans(
    X_weighted.T, num_clusters, fuzziness_parameter, error=.005, maxiter=1000
)


# Convert fuzzy to hard clusters
hard_labels = np.argmax(u, axis=0)
print(f"Hard cluster assignments: {len(hard_labels)} customers")
print(f"Cluster distribution: {np.bincount(hard_labels)}")


Running Fuzzy C-Means with 6 clusters...
Fuzziness parameter: 2
Hard cluster assignments: 1070 customers
Cluster distribution: [ 21 543   4 426  69   7]


In [35]:
# Revenue-based cluster ordering (same as original model)
print("Ordering clusters by average revenue...")

# Calculate weighted average revenue for each cluster
avg_revenue = []
for i in range(num_clusters):
    membership = u[i, :]
    revenue = X_scaled[:, 0]  # Assuming first feature is revenue-related
    if membership.sum() > 0:
        cluster_avg = np.average(revenue, weights=membership)
    else:
        cluster_avg = 0
    avg_revenue.append(cluster_avg)

avg_revenue = np.array(avg_revenue)
print(f"Average revenue per cluster: {avg_revenue}")

# Sort clusters by average revenue
order = np.argsort(avg_revenue)
mapping = np.empty_like(order)
mapping[order] = np.arange(num_clusters)

# Apply mapping to get ordered labels
renumbered_labels = mapping[hard_labels]
print(renumbered_labels)
# Create meaningful cluster names
cluster_names = {0: 'Rock', 1: 'Bronze', 2: 'Silver', 3: 'Gold', 4: 'Diamond', 5: 'Platinum'}
renumbered_labels_names = np.vectorize(cluster_names.get)(renumbered_labels)

# Add cluster labels directly to the original data
customer_data_clean['Cluster'] = renumbered_labels_names
customer_data_clean['Cluster_Numeric'] = renumbered_labels

Ordering clusters by average revenue...
Average revenue per cluster: [-0.36919159  0.03660354 -0.49958752  0.00930881 -0.11740854 -0.47389731]
[5 5 4 ... 5 4 4]


In [37]:
print("Customer segmentation completed!")
print(f"Total customers segmented: {len(customer_data_clean)}")

# Debug: Check for duplicate columns
print(f"\nDataFrame columns: {list(customer_data_clean.columns)}")
print(f"Total columns: {len(customer_data_clean.columns)}")

# Check for duplicate column names
duplicated_mask = customer_data_clean.columns.duplicated()
duplicated_columns = customer_data_clean.columns[duplicated_mask]
print(f"Duplicate columns: {list(duplicated_columns)}")

# Show all column names with their counts
from collections import Counter
column_counts = Counter(customer_data_clean.columns)
duplicated_with_counts = {col: count for col, count in column_counts.items() if count > 1}
print(f"Columns with multiple occurrences: {duplicated_with_counts}")

print(f"\nCluster column type: {type(customer_data_clean['Cluster'])}")
print(f"Cluster column shape: {customer_data_clean['Cluster'].shape if hasattr(customer_data_clean['Cluster'], 'shape') else 'No shape attribute'}")

# Check if Cluster column exists and is 1-dimensional
if 'Cluster' in customer_data_clean.columns:
    cluster_col = customer_data_clean['Cluster']
    
    # Try to fix if it's multi-dimensional
    if hasattr(cluster_col, 'ndim') and cluster_col.ndim > 1:
        print("Warning: Cluster column is multi-dimensional. Attempting to fix...")
        customer_data_clean['Cluster'] = cluster_col.iloc[:, 0] if cluster_col.ndim > 1 else cluster_col
    
print("\nCluster distribution:")
cluster_counts = customer_data_clean['Cluster'].value_counts().sort_index()
for cluster, count in cluster_counts.items():
    print(f"  {cluster}: {count} customers ({count/len(customer_data_clean)*100:.1f}%)")

# Display sample customers from each cluster
print("\nSample customers from each cluster:")
for cluster in ['Rock', 'Bronze', 'Silver', 'Gold', 'Diamond', 'Platinum']:
    cluster_data = customer_data_clean[customer_data_clean['Cluster'] == cluster]
    if len(cluster_data) > 0:
        print(f"\n{cluster} customers (sample):")
        sample = cluster_data[['CustomerID', 'TotalProfit', 'ProfitMargin', 'AvgOrderValue']].head(3)
        print(sample.to_string(index=False))


Customer segmentation completed!
Total customers segmented: 1070

DataFrame columns: ['CustomerID', 'ExtPrice_sum', 'ExtPrice_mean', 'ExtPrice_std', 'ExtPrice_count', 'ExtPrice_median', 'ExtPrice_min', 'ExtPrice_max', 'ExtCost_sum', 'ExtCost_mean', 'ExtCost_std', 'ExtCost_median', 'SalesQty_sum', 'SalesQty_mean', 'SalesQty_std', 'SalesQty_median', 'SalesQty_min', 'SalesQty_max', 'UnitPrice_mean_x', 'UnitPrice_std_x', 'UnitPrice_min_x', 'UnitPrice_max_x', 'UnitPrice_median', 'UnitCost_mean', 'UnitCost_std', 'UnitCost_median', 'OrderNumber_nunique', 'ProductID_nunique', 'SalesDate_min', 'SalesDate_max', 'SalesDate_count', 'TotalRevenue', 'TotalCost', 'TotalProfit', 'ProfitMargin', 'AvgOrderValue', 'AvgOrderQuantity', 'PriceVolatility', 'QuantityVolatility', 'CustomerLifetimeDays', 'AvgDaysBetweenOrders', 'OrderFrequency', 'ProductDiversity', 'AvgProductsPerOrder', 'Recency', 'Frequency', 'Monetary', 'RecencyScore', 'FrequencyScore', 'MonetaryScore', 'RFMScore', 'CustomerLifetimeValue', '

## Step 6: Model Evaluation


In [38]:
# Calculate clustering quality metrics
print("Calculating clustering quality metrics...")

# Silhouette Score (higher is better, range: -1 to 1)
silhouette_avg = silhouette_score(X_weighted, renumbered_labels)
print(f"Silhouette Score: {silhouette_avg:.4f}")

# Davies-Bouldin Index (lower is better)
davies_bouldin = davies_bouldin_score(X_weighted, renumbered_labels)
print(f"Davies-Bouldin Index: {davies_bouldin:.4f}")

# Calinski-Harabasz Index (higher is better)
calinski_harabasz = calinski_harabasz_score(X_weighted, renumbered_labels)
print(f"Calinski-Harabasz Index: {calinski_harabasz:.4f}")

print("\nInterpretation:")
print(f"- Silhouette Score: {silhouette_avg:.4f} ({'Good' if silhouette_avg > 0.5 else 'Fair' if silhouette_avg > 0.3 else 'Poor'})")
print(f"- Davies-Bouldin Index: {davies_bouldin:.4f} ({'Good' if davies_bouldin < 1.0 else 'Fair' if davies_bouldin < 2.0 else 'Poor'})")
print(f"- Calinski-Harabasz Index: {calinski_harabasz:.4f} ({'Good' if calinski_harabasz > 100 else 'Fair' if calinski_harabasz > 50 else 'Poor'})")


Calculating clustering quality metrics...
Silhouette Score: 0.3512
Davies-Bouldin Index: 0.9105
Calinski-Harabasz Index: 817.1281

Interpretation:
- Silhouette Score: 0.3512 (Fair)
- Davies-Bouldin Index: 0.9105 (Good)
- Calinski-Harabasz Index: 817.1281 (Good)


## Step 7: Save the Model


In [39]:
# Save the model components
model_data = {
    'centers': center,
    'mapping': mapping,
    'cluster_names': cluster_names,
    'fuzziness_parameter': fuzziness_parameter,
    'num_clusters': num_clusters,
    'weights': weights,
    'features': features,
    'scaler': scaler
}

# Save to a file
with open('fuzzy_cmeans_customer_model.pkl', 'wb') as f:
    pickle.dump(model_data, f)

print("Model saved successfully!")
print("Saved components:")
print("- Cluster centers")
print("- Cluster mapping")
print("- Cluster names")
print("- Feature weights")
print("- Feature list")
print("- Scaler for preprocessing")

# Save the segmented customer data
customer_data_clean.to_csv('segmented_customers.csv', index=False)
print("\nSegmented customer data saved to 'segmented_customers.csv'")


Model saved successfully!
Saved components:
- Cluster centers
- Cluster mapping
- Cluster names
- Feature weights
- Feature list
- Scaler for preprocessing

Segmented customer data saved to 'segmented_customers.csv'


## Summary

This notebook implements customer segmentation using:

1. **Pure Correlation Method**: Feature weights determined by statistical correlations with TotalProfit
2. **Fuzzy C-Means Clustering**: Creates 6 customer segments with meaningful names
3. **Revenue-Based Ordering**: Clusters ordered from Rock (lowest) to Platinum (highest)
4. **Model Evaluation**: Quality metrics to assess clustering performance
5. **Model Persistence**: Saves model for future use

### Key Features Used:
- ProfitMargin, AvgOrderValue, CustomerLifetime
- QuoteConversionRate, OrderNumber_nunique, ProductID_nunique, PriceVolatility

### Cluster Names:
- **Rock**: Lowest value customers
- **Bronze**: Low-margin customers  
- **Silver**: Mid-value customers
- **Gold**: High-value customers
- **Diamond**: Premium customers
- **Platinum**: Top-tier customers

The model is now ready for business applications and customer strategy development.
